# Personal Financial Assistant

## Objective

This notebook demonstrates the following:

- Function Calling with Yfinance to get latest stock prices. Summarization of user provided article. Extract country info from article, extract country, capital and other aspects, and call an API to get more information about each country.

This tutorial uses the following Azure AI services:
- Access to Azure OpenAI Service - you can apply for access [here](https://aka.ms/oai/access)
- Azure OpenAI service - you can create it from instructions [here](https://learn.microsoft.com/en-us/azure/ai-services/openai/how-to/create-resource)
- Azure OpenAI Studio - go to [https://oai.azure.com/](https://oai.azure.com/) to work with the Assistants API Playground
- A connection to the Azure OpenAI Service with a [Key and Endpoint](https://learn.microsoft.com/en-us/azure/ai-services/openai/chatgpt-quickstart)

Reference:
- Learn more about how to use Assistants with our [How-to guide on Assistants](https://learn.microsoft.com/en-us/azure/ai-services/openai/how-to/assistant)
- [Assistants OpenAI Overview](https://platform.openai.com/docs/assistants/overview)

## Time

You should expect to spend 10-15 minutes running this sample. 

## About this example

The objective of the provided Python file is to create an Azure OpenAI Assistant named "Portfolio Management Assistant" using the Azure OpenAI API. The assistant is designed to act as a personal financial assistant, providing information and insights related to a user's investment portfolio. The script initiates a conversation with the assistant, guiding it through various financial queries and scenarios to showcase its capabilities.

### Data
This sample uses files from the folder [`data/`](./data/) in this repo. You can clone this repo or copy this folder to make sure you have access to these files when running the sample.



## Before you begin



### Installation

Install the following packages required to execute this notebook. 



In [1]:
# Install the packages
%pip install --upgrade pip
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### Parameters

In [2]:
import os

from dotenv import load_dotenv

load_dotenv("../.env")
api_endpoint = os.getenv("OPENAI_URI")
api_key = os.getenv("OPENAI_KEY")
api_version = os.getenv("OPENAI_VERSION")
api_deployment_name = os.getenv("OPENAI_GPT_DEPLOYMENT")
email_URI = os.getenv("EMAIL_URI")

should_cleanup: bool = True

## Run this Example

### Load the required libraries

In [3]:
import html
import sys
from datetime import datetime
from pathlib import Path
import json

import requests
import yfinance as yf
from openai import AzureOpenAI
from openai.types import FileObject
from openai.types.beta import Thread
from openai.types.beta.threads import Run

sys.path.append(str(Path().resolve().parent / "common"))
from common import Common

### Create an AzureOpenAI client

In [4]:
client = AzureOpenAI(api_key=api_key, api_version=api_version, azure_endpoint=api_endpoint)

### Get the latest stock price by ticker symbol using Yahoo Finance

In [5]:
def get_stock_price(symbol: str) -> float:
    stock = yf.Ticker(symbol)
    return stock.history(period="1d")["Close"].iloc[-1]

### Send an email using Logic Apps

In [6]:
def send_logic_apps_email(to: str, content: str) -> None:
    json_payload = {"to": to, "content": html.unescape(content)}
    headers = {"Content-Type": "application/json"}
    response = requests.post(email_URI, json=json_payload, headers=headers)
    if response.status_code == 202:
        print("Email sent to: " + json_payload["to"])

### Define the Assistant tools

In [7]:
tools_list = [
    {"type": "code_interpreter"},
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Retrieve the latest closing price of a stock using its ticker symbol.",
            "parameters": {
                "type": "object",
                "properties": {"symbol": {"type": "string", "description": "The ticker symbol of the stock"}},
                "required": ["symbol"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": "Sends an email to a recipient(s).",
            "parameters": {
                "type": "object",
                "properties": {
                    "to": {"type": "string", "description": "The email(s) the email should be sent to."},
                    "content": {"type": "string", "description": "The content of the email."},
                },
                "required": ["to", "content"],
            },
        },
    },
]

### Upload the file(s)

In [8]:
DATA_FOLDER = "data/"

def upload_file(client: AzureOpenAI, path: str) -> FileObject:
    with Path(path).open("rb") as f:
        return client.files.create(file=f, purpose="assistants")


arr = os.listdir(DATA_FOLDER)
assistant_files = []
for file in arr:
    filePath = DATA_FOLDER + file
    assistant_files.append(upload_file(client, filePath))

file_ids = [file.id for file in assistant_files]

### Create an Assistant and a Thread

In [9]:
assistant = client.beta.assistants.create(
    name="Portfolio Management Assistant",
    instructions="You are a personal securities trading assistant. Please be polite, professional, helpful, and friendly. "
    + "Use the provided portfolio CSV file to answer the questions. If question is not related to the portfolio or you cannot answer the question, say, 'contact a representative for more assistance.'"
    + "If the user asks for help or says 'help', provide a list of sample questions that you can answer.",
    tools=tools_list,
    model=api_deployment_name,
    tool_resources={  # Optional
        "code_interpreter": {
            "file_ids": file_ids  # Optional
        }
    }
)

thread = client.beta.threads.create()

### Process Function calling

In [10]:
def call_functions(client: AzureOpenAI, thread: Thread, run: Run) -> None:


    required_actions = run.required_action.submit_tool_outputs.model_dump()
    tool_outputs = []

    for action in required_actions["tool_calls"]:
        func_name = action["function"]["name"]
        arguments = json.loads(action["function"]["arguments"])

        if func_name == "get_stock_price":
            output = get_stock_price(symbol=arguments["symbol"])
            tool_outputs.append({"tool_call_id": action["id"], "output": str(output)})
        elif func_name == "send_email":
            print("Sending email...")
            print("Email content:")
            print(arguments["content"])
            tool_outputs.append({"tool_call_id": action["id"], "output": "Email sent"})
        else:
            raise ValueError(f"Unknown function: {func_name}")

    client.beta.threads.runs.submit_tool_outputs(thread_id=thread.id, run_id=run.id, tool_outputs=tool_outputs)

### Have a conversation with the Assistant

In [11]:
common = Common(client, assistant, thread)

instructions="The current date and time is: " + datetime.now().strftime("%x %X") + "."

In [12]:
common.process_message(instructions, "Based on the provided portfolio, what investments do I own?", call_functions)

processing ...
assistant:
Based on the provided portfolio, you own the following investments:

1. Microsoft (MSFT)
2. Apple (AAPL)
3. Amazon (AMZN)
4. Tesla (TSLA)
5. Netflix (NFLX)

If there's anything specific you would like to know about these investments, please feel free to ask!



In [13]:
common.process_message(instructions, "What is the value of my portfolio?", call_functions)

processing ...
assistant:
Based on the provided portfolio, you own the following investments:

1. Microsoft (MSFT)
2. Apple (AAPL)
3. Amazon (AMZN)
4. Tesla (TSLA)
5. Netflix (NFLX)

If there's anything specific you would like to know about these investments, please feel free to ask!

assistant:
It looks like there is an error due to an additional row in the portfolio dataframe. Let me fix that and calculate the total portfolio value for you.

assistant:
The total value of your portfolio is $262,500.70.

If you have any further questions or if there's anything else I can assist you with regarding your portfolio, feel free to ask!



In [14]:
common.process_message(instructions, "What is my best and worst investment?", call_functions)

processing ...
assistant:
Based on the provided portfolio, you own the following investments:

1. Microsoft (MSFT)
2. Apple (AAPL)
3. Amazon (AMZN)
4. Tesla (TSLA)
5. Netflix (NFLX)

If there's anything specific you would like to know about these investments, please feel free to ask!

assistant:
It looks like there is an error due to an additional row in the portfolio dataframe. Let me fix that and calculate the total portfolio value for you.

assistant:
The total value of your portfolio is $262,500.70.

If you have any further questions or if there's anything else I can assist you with regarding your portfolio, feel free to ask!

assistant:
Based on the value change from the average cost, your best investment is in Microsoft (MSFT) with a profit of $72,111.00. However, your worst investment is in Tesla (TSLA) with a loss of $65,077.00.

If you have any further questions or need additional information, feel free to ask!



In [15]:
common.process_message(instructions, "Chart the realized gain or loss of my investments.", call_functions)

processing ...
assistant:
Based on the provided portfolio, you own the following investments:

1. Microsoft (MSFT)
2. Apple (AAPL)
3. Amazon (AMZN)
4. Tesla (TSLA)
5. Netflix (NFLX)

If there's anything specific you would like to know about these investments, please feel free to ask!

assistant:
It looks like there is an error due to an additional row in the portfolio dataframe. Let me fix that and calculate the total portfolio value for you.

assistant:
The total value of your portfolio is $262,500.70.

If you have any further questions or if there's anything else I can assist you with regarding your portfolio, feel free to ask!

assistant:
Based on the value change from the average cost, your best investment is in Microsoft (MSFT) with a profit of $72,111.00. However, your worst investment is in Tesla (TSLA) with a loss of $65,077.00.

If you have any further questions or need additional information, feel free to ask!

assistant:
Here is a bar chart showing the realized gain or loss 

In [16]:
common.process_message(
    instructions,
    "Please send a report to name@contoso.com with the details for each stock based on the latest stock prices, and list the best and worst performing stocks in my portfolio.",
    call_functions,
)

processing ...
Sending email...
Email content:
Dear Investor,

Here is a summary of your stock portfolio based on the latest stock prices:

- Microsoft (MSFT): $440.37
- Apple (AAPL): $224.18
- Amazon (AMZN): $183.75
- Tesla (TSLA): $249.23
- Netflix (NFLX): $643.04

Best Performing Stock:
- Symbol: MSFT, Realized Gain: $72,111.00

Worst Performing Stock:
- Symbol: TSLA, Realized Loss: $65,077.00

If you have any further questions or need assistance, please don't hesitate to reach out.

Thank you,
Your Financial Assistant
assistant:
Based on the provided portfolio, you own the following investments:

1. Microsoft (MSFT)
2. Apple (AAPL)
3. Amazon (AMZN)
4. Tesla (TSLA)
5. Netflix (NFLX)

If there's anything specific you would like to know about these investments, please feel free to ask!

assistant:
It looks like there is an error due to an additional row in the portfolio dataframe. Let me fix that and calculate the total portfolio value for you.

assistant:
The total value of your port

## Cleaning up

In [17]:
if should_cleanup:
    client.beta.assistants.delete(assistant.id)
    client.beta.threads.delete(thread.id)
    for file in assistant_files:
        client.files.delete(file.id)